# CAND-VB2 RAG — Google Colab

Trợ lý RAG chuyên biệt về **VB2CA tuyển mới năm 2026** dành cho công dân đã có bằng đại học, ưu tiên các câu hỏi của người có văn bằng 1 CNTT/IT.

- Dataset mặc định: snapshot nguồn chính thức đã xác minh đến **08/09/2026**.
- External AI APIs: **Gemini + OpenRouter only**.
- `providers.env` không nằm trên GitHub; notebook kiểm tra `/content/providers.env` và yêu cầu upload nếu thiếu.


In [ ]:
# 1) Clone / update repository
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/NVTruong473/NLP.git"
REPO_DIR = Path("/content/NLP")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# 2) Install dependencies
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps


## 3) Load private API keys

Tạo file `providers.env` từ `providers.env.example`. Không commit key thật lên GitHub.


In [ ]:
from pathlib import Path
import shutil

ENV_PATH = Path("/content/providers.env")
drive_candidate = Path("/content/drive/MyDrive/providers.env")

if not ENV_PATH.exists() and drive_candidate.exists():
    shutil.copy2(drive_candidate, ENV_PATH)

if not ENV_PATH.exists():
    from google.colab import files
    print("Không thấy /content/providers.env. Hãy upload file providers.env riêng tư của bạn.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No providers.env uploaded.")
    selected_name = next(iter(uploaded))
    Path(selected_name).replace(ENV_PATH)

from vietrag.secrets import load_provider_env, key_summary
load_provider_env(ENV_PATH)
print("Credentials loaded (values hidden):", key_summary())


## 4) Inspect the verified dataset

Colab mặc định dùng **local curated snapshots** trong `data/corpus/`, không crawl web mỗi lần chạy. Nguồn gốc nằm ở `data/source_registry.yaml`.


In [ ]:
from pathlib import Path
import yaml

registry = yaml.safe_load(Path("data/source_registry.yaml").read_text(encoding="utf-8"))
print("Dataset:", registry["dataset"]["name"])
print("Verified at:", registry["dataset"]["verified_at"])
print("Official sources:", len(registry["sources"]))
for s in registry["sources"]:
    print(f"- {s['id']}: {s['authority']} | {s['status']}")


## 5) Build the index

Gemini tạo embedding; FAISS + BM25 chạy local. Index được giữ trong runtime để không tốn quota embedding lại nếu cell được chạy lại.


In [ ]:
import subprocess
from pathlib import Path

INDEX_FILE = Path("artifacts/index/faiss.index")
REBUILD_INDEX = False

if REBUILD_INDEX or not INDEX_FILE.exists():
    subprocess.run([
        "python", "scripts/build_index.py",
        "--input-dir", "data/corpus",
        "--env", str(ENV_PATH),
    ], check=True)
else:
    print("Existing index found; skip re-embedding.")


## 6) Chat with evidence

Mỗi khẳng định quan trọng phải có `[S#]`; dưới câu trả lời sẽ hiển thị URL chính thức, cơ quan ban hành và ngày xác minh.


In [ ]:
from vietrag.config import load_config
from vietrag.pipeline import VietRAGPipeline
from vietrag.providers import GeminiProvider, OpenRouterProvider

cfg = load_config("configs/default.yaml")
rag = VietRAGPipeline.load(cfg, GeminiProvider(), OpenRouterProvider())

question = "Tôi có bằng đại học CNTT loại Khá thì có thể đi theo hướng VB2 Công an nào trong năm 2026?"
result = rag.ask(question)
print(result.text)
print(rag.format_sources(result))
print("\nprovider=", result.provider, "top_dense=", round(result.top_dense_score, 4), "refused=", result.refused)


### Kiểm tra câu hỏi phụ thuộc thời gian


In [ ]:
q = "Hôm nay 8/9/2026 tôi chưa đăng ký sơ tuyển thì còn bắt đầu hồ sơ VB2CA 2026 được không?"
r = rag.ask(q)
print(r.text)
print(rag.format_sources(r))


### Kiểm tra out-of-domain


In [ ]:
r = rag.ask("Hãy cho tôi công thức làm bánh tiramisu.")
print(r.text)
print("refused=", r.refused)


## 7) Evaluate retrieval + OOD


In [ ]:
!python scripts/evaluate.py --env /content/providers.env


## 8) Calibrate OOD threshold

Threshold mặc định chỉ là điểm bắt đầu. Dùng tập benchmark hiện tại để xem ngưỡng đề xuất, rồi mở rộng benchmark trước khi báo cáo kết quả chính thức.


In [ ]:
!python scripts/calibrate_ood.py --env /content/providers.env


## 9) Launch Gradio chat app


In [ ]:
!python app.py


## Optional: live-source experiment

Nếu muốn thử crawl lại website chính thức tại thời điểm chạy, có thể build riêng bằng `data/official_sources.yaml`. Không dùng cách này làm benchmark mặc định vì nội dung web có thể thay đổi theo thời gian.

```bash
python scripts/build_index.py --sources data/official_sources.yaml --input-dir data/corpus --env /content/providers.env
```
